![](./mira/gui/resources/pics/Logo_Sykno.svg "")

# __MiRa Data Evaluation__ - ___Jupyter Notebook___

### __Introduction__


**MiRa Data Measurement:**

- MiRa Measurements Path (HDF5 Files): `/mira_root/mira/meas/hdf5/`
- MiRa HDF5 Controller for interaction via MiRa Eval GUI or within the Jupyter Notebook: `/mira_root/mira/meas/mira_hdf5_ctrl.py` (see MiRa HDF5 Controller Usage)
- The MiRa radar data is a Numpy array `radar_data_cube` with a shape of `(n_frames, n_samples_per_chirp, n_rx, n_tx, n_shape_sets)` and its `dtype` is `float32`.


**MiRa HDF5 Structure:**
- __/__
  - __Data__
    - ***Frame_Data_Cube_XXXX_ZZZZ***
      - `@delta_time`: `np.float32`
      - `@shape`: `(n_samples_per_chirp, n_rx, n_tx, n_shape_sets)`
      - `@timestamp`: `np.float32`
  - __Metadata__
    - ***mira_bgt_reg_content***
      - `@mira_bgt_reg_content`:
    - ***mira_bgt_reg_content_readable***
      - `@mira_bgt_reg_content_readable`:
    - ***mira_config***
      - `@mira_config`:
    - ***mira_radar_parameters*** 
      - `mira_radar_parameters`

***Frame_Data_Cube_XXXX_ZZZZ*** where ***XXXX*** are multiples of `4095` and ***ZZZZ*** is the frame counter with values in the range $[0, 4095]$.

- ***XXXX*** $\in \{k \cdot 4095 \mid k \in \mathbb{N}, 0 \leq k \geq 9999\}$
- ***ZZZZ*** $\in \{0, 1, 2, \ldots, 4095\}$


**Working with HDF5:**

- List measurement structure:
```sh 
    h5ls -r mira_hdf5_filename.hdf5
```
- Inspect a specific group or dataset:
```sh
    h5dump -d /Data/Frame_Data_Cube_XXXX_ZZZZ mira_hdf5_filename.hdf5
```
- Inspect measurement meta data:
```sh
    h5dump -g /Metadata mira_hdf5_filename.hdf5
```
- Get statistics e.g. number of groups, datasets, and attributes:
```sh
    h5stat mira_hdf5_filename.hdf5
```


In [ ]:
description = "MiRa Root is: "
mira_root = !pwd
print(f"{description}{mira_root[0]}")
mira_meas_hdf5_path = f'{mira_root[0]}/mira/meas/hdf5/'
!tree -C {mira_meas_hdf5_path}

In [ ]:
mira_meas_hdf5_file_path = f'{mira_meas_hdf5_path}/MiRa6024_Measurement_08_02_2024_16_59_45_Tescht_Tescht.hdf5'

In [ ]:
!h5ls -r {mira_meas_hdf5_file_path}

In [ ]:
!h5dump -d /Data/Frame_Data_Cube_0000_0001 {mira_meas_hdf5_file_path}

In [ ]:
!h5dump -g /Metadata {mira_meas_hdf5_file_path}

In [ ]:
!h5stat {mira_meas_hdf5_file_path}

### __MiRa Data Evaluation__ - ___Python Code___


#### ***MiRa HDF5 Controller Usage***


In [ ]:
import os
import ipywidgets as widgets
from mira.meas.mira_hdf5_ctrl import MIRA_HDF5_CTRL
mira_meas_hdf5_path = './mira/meas/hdf5/'

# List all files in the folder
file_names = os.listdir(mira_meas_hdf5_path)

# Filter out non-file entries if needed
file_names = [f for f in file_names if os.path.isfile(os.path.join(mira_meas_hdf5_path, f)) and f != '.gitkeep']
sorted_file_names = sorted(file_names)

# Dropdown for file selection
file_dropdown = widgets.Dropdown(
    options=file_names,
    description=f'Select a File:',
)
print((f'MiRa HDF5 Files in {mira_meas_hdf5_path}:'))
file_dropdown.layout.width = '650px'  # Adjust the width as needed
display(file_dropdown)


In [ ]:
mira_hdf5_controller = MIRA_HDF5_CTRL(mira_meas_hdf5_path+file_dropdown.value)
data = mira_hdf5_controller.read_dataset("/Data/Frame_Data_Cube_0000_0002")
print(data)

#### ***Inspect MiRa Dataset***

1. **Load python packages and MiRa HDF5 measurement file**


In [ ]:
import h5py
import pickle
import numpy as np
from mira.rsys.mira_radar_sys import MIRA_RADAR_PARAMETER

file_path = './mira/meas/hdf5/MiRa6024_Measurement_08_02_2024_16_59_45_Tescht_Tescht.hdf5'

def print_hdf5_item_structure(file, itempath='/'):
    for item_name in file[itempath]:
        path = f'{itempath}/{item_name}' if itempath != '/' else f'/{item_name}'
        if isinstance(file[itempath + '/' + item_name], h5py.Group):
            print_hdf5_item_structure(file, path)

2. **Open MiRa HDF5 measurement file, load meta data and build radar data cube**

In [ ]:
with h5py.File(file_path, 'r') as file:
    print_hdf5_item_structure(file)
    data_group = file['Data']

    datasets = []
    for dataset_name in file['/Data']:
        data = np.array(file['/Data'][dataset_name][:], dtype=np.float32)
        datasets.append(np.expand_dims(data, axis=0))

    mira_data_cube = np.concatenate(datasets, axis=0, dtype=np.float32) 
    
    # Reading a specific dataset
    frame_data_cube = data_group['Frame_Data_Cube_0000_0001']
    delta_time = frame_data_cube.attrs['delta_time']
    shape = frame_data_cube.attrs['shape']
    timestamp = frame_data_cube.attrs['timestamp']
    frame_data_cube = np.array(frame_data_cube, dtype=np.float32)

    # Accessing the 'Metadata' group
    metadata_group = file['Metadata']

    # Reading 'mira_config'
    mira_config = metadata_group['mira_config'].attrs['mira_config']

    # Reading 'mira_bgt_reg_content'
    mira_bgt_reg_content = metadata_group['mira_bgt_reg_content'].attrs['mira_bgt_reg_content']

    # Reading 'mira_bgt_reg_content_readable'
    mira_bgt_reg_content_readable = metadata_group['mira_bgt_reg_content_readable'].attrs['mira_bgt_reg_content_readable']

    # Example: Reading content from 'mira_radar_parameters'
    mira_radar_parameters = metadata_group['mira_radar_parameters']['mira_radar_parameters']

    # Retrieve the numpy.void object and convert it back to bytes
    radar_param_data = mira_radar_parameters[()].tobytes()
    
    # Unpickle the bytes to get back the original Python object
    radar_param: MIRA_RADAR_PARAMETER = pickle.loads(radar_param_data)
    


3. **MiRa HDF5 measurement - data profiling**

    __Inspect MiRa Radar System Parameters__
    - __radar_param__: meta-class contains __All__ other sub-classes
        - __sys__: sub-class contains __Radar__ specific values
        - __dsp__: sub-class contains __DSP__ parameters
        - __mon__: sub-class to __Monitor__ frame count, duration time, etc.
        - __gui__: sub-class contains __GUI__ relevant parameters
        - __meas__: sub-class contains __Measurement__ relevant parameters
        - __remt__: sub-class contains __Remote Control__ relevant parameters
        - __rply__: sub-class contains __Replay Control__ relevant parameters 

In [ ]:
print(f"{radar_param=}")
print(f"{radar_param.dsp=}")
print(f"{radar_param.gui=}")
print(f"{radar_param.mon=}")
print(f"{radar_param.sys=}")
print(f"{radar_param.meas=}")
print(f"{radar_param.remt=}")
print(f"{radar_param.rply=}")

3. **MiRa HDF5 measurement - data profiling**

    2. __Check MiRa Radar System Parameters__
    - __radar_param__: meta-class contains __All__ other sub-classes

In [ ]:
print(f"{mira_data_cube.shape=}")
print(f"{frame_data_cube.shape=}\n{delta_time=}\n{shape=}\n{timestamp=}")

print(f"{frame_data_cube[:8,:,:,0]}")

# print(f"{mira_bgt_reg_content=}")
# print(f"{mira_bgt_reg_content_readable=}")
print(f"{mira_radar_parameters=}")
file.close()

#### ***Postprocessing MiRa Data***

1. **Load MiRa Processing Module and create a Data Processor**

In [ ]:
from mira.proc.mira_data_processing import MIRA_DATA_PROCESSOR
mira_data_processor = MIRA_DATA_PROCESSOR(radar_param)

2. **Time Singal Processing**

In [ ]:
scaled_raw_data = mira_data_processor.scale_raw_data(mira_data_cube[1,...], preprocessing=False, output_mean=False)
print(f"{scaled_raw_data.shape=}")
scaled_raw_data_mean = mira_data_processor.scale_raw_data(mira_data_cube[1,...], preprocessing=False, output_mean=True)
print(f"{scaled_raw_data_mean.shape=}")
scaled_raw_data_preprocessed = mira_data_processor.scale_raw_data(mira_data_cube[1,...], preprocessing=True, output_mean=False)
print(f"{scaled_raw_data_preprocessed.shape=}")
scaled_raw_data_preprocessed_mean = mira_data_processor.scale_raw_data(mira_data_cube[1,...], preprocessing=True, output_mean=True)
print(f"{scaled_raw_data_preprocessed_mean.shape=}")

2. **Spectrum Singal Processing**

In [ ]:
# TODO

3. **Spectrogram Signal Processing**


In [ ]:
for frame in range(mira_data_cube.shape[0]-1):
    spectrogram_map = mira_data_processor._calc_spectogram(mira_data_cube[frame+1,:,:,0,:], 0)
print(spectrogram_map)

4. **Range Doppler Signal Processing**

In [ ]:
# TODO

5. **Range Azimuth Signal Processing**

In [ ]:
# TODO

In [ ]:
# TODO Implement some interactive features to jupyter notebook
import ipywidgets as widgets
from IPython.display import display
import datetime
import matplotlib.pyplot as plt
import numpy as np

# Example datasets
datasets = {
    'Sine Wave': np.sin,
    'Cosine Wave': np.cos
}

# Dropdown for dataset selection
dataset_dropdown = widgets.Dropdown(
    options=list(datasets.keys()),
    value='Sine Wave',
    description='Dataset:',
)

# Slider for frequency adjustment
frequency_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=5.0,
    step=0.1,
    description='Frequency:',
)

# Function to update plot
def update_plot(dataset_name, frequency):
    plt.clf() # Clear the current figure
    func = datasets[dataset_name] # Get the function to use
    x = np.linspace(0, 2 * np.pi, 1000)
    y = func(frequency * x)
    plt.plot(x, y)
    plt.show()

# Interactive widget
widgets.interactive_output(update_plot, {'dataset_name': dataset_dropdown, 'frequency': frequency_slider})

# Display widgets
display(dataset_dropdown, frequency_slider)

# Button widget
button = widgets.Button(description="Click Me!")
def on_button_clicked(b):
    print("Button clicked.")

button.on_click(on_button_clicked)

# Accordion widget
accordion = widgets.Accordion(children=[widgets.IntSlider(), widgets.Text()])
accordion.set_title(0, 'Slider')
accordion.set_title(1, 'Text')

display(button, accordion)
